# Stage 03b — Per-animal frozen clustering configuration

Loads exactly one Stage 03a **open-set** `.npy` (query-query plus query-gallery). Attach that 03a output dataset only — images and the splits CSV are already inside the artifact. Set `DATASET_NAME` and `GLOBAL_DESCRIPTOR` to match the file `sim_matrix_{GLOBAL_DESCRIPTOR}_{DATASET_NAME}_open.npy`. Do not rebuild the split.

- **Standard datasets:** run 2,000 TPE trials on those sealed qq/qd matrices (known-identity attachment, re-ranking, Chinese Whispers). Do not rebuild a gallery/query split here.
- **Gallery-free animals** (Jaguar, or any dataset whose split has no gallery): skip Optuna and freeze documented query-query clustering defaults; no database scoring or known-ID attachment parameters are used. Protocol is read from the 03a artifact, not hardcoded by dataset name.

Local layout (so other models do not overwrite this one):
- matrix in: `outputs/stage03/03a/{GLOBAL_DESCRIPTOR}/{DATASET_NAME}/`
- writes to: `outputs/stage03/03b/{GLOBAL_DESCRIPTOR}/{DATASET_NAME}/best_configs_….json`

The output is `best_configs_{GLOBAL_DESCRIPTOR}_{DATASET_NAME}.json` for notebook 03c.


In [7]:
%pip install -q optuna networkx scikit-learn

from collections import defaultdict
import hashlib
import json
import os
import random
from pathlib import Path

import networkx as nx
import numpy as np
import optuna
import pandas as pd
from sklearn.metrics import adjusted_rand_score, v_measure_score

# ============================================================
# USER KNOBS — must match the 03a artifact
# ============================================================
DATASET_NAME = "bristol_gorillas"
GLOBAL_DESCRIPTOR = "ApexID_BHCA_A_v2"
N_TRIALS = 2000
OPTUNA_SEED = 42
SPLIT_SEED = 42
KNOWN_IDENTITY_FRAC = 0.8
DB_FRAC_WITHIN_KNOWN = 0.8
MIN_IMAGES_PER_KNOWN_IDENTITY = 3
# Match the AnimalCLEF objective: reranking is always enabled and its parameters are tuned.
USE_RERANK = True

PROTOCOL = None  # overwritten from the 03a artifact after load
STAGE03_SPLITS_ROOT_CANDIDATES = (
    Path(os.environ.get("APEXID_STAGE03_SPLITS", "")),
    Path("/kaggle/input/datasets/mohamedbassat/03-open-set-splits-data/apexid_cvpr_stage03/03_splits"),
    Path("/kaggle/input/03-open-set-splits-data/apexid_cvpr_stage03/03_splits"),
    Path("/kaggle/working/apexid_cvpr_stage03/03_splits"),
)
QUERY_ONLY_DEFAULTS = {
    # Frozen before Jaguar evaluation; no Jaguar labels are used to select these.
    "cw_threshold": 0.30,
    "cw_top_k": 20,
    "cw_iterations": 20,
    "cw_k_core_degree": 1,
    "use_rerank": True,
    "rr_k1": 20,
    "rr_k2": 6,
    "rr_lambda": 0.30,
}

ALLOWED_DATASETS = ("bristol_gorillas", "spac_wild", "elephants", "gcn_id", "jaguar_antonio")
ALLOWED_GLOBAL_DESCRIPTORS = (
    "ApexID_BHCA_A_v2", "ApexID_BHCA_SCA_v2", "ApexID_BHCA_SCSF2_v2", "MiewIDv3", "MegaDescriptor",
)
assert DATASET_NAME in ALLOWED_DATASETS
assert GLOBAL_DESCRIPTOR in ALLOWED_GLOBAL_DESCRIPTORS

artifact_name = f"sim_matrix_{GLOBAL_DESCRIPTOR}_{DATASET_NAME}_open.npy"
ON_KAGGLE = Path("/kaggle/working").is_dir()
HERE = Path.cwd().resolve()
REPO_ROOT = HERE.parent if HERE.name == "notebooks" else HERE
STAGE03A_MATRIX_CANDIDATES = [
    Path(os.environ.get("APEXID_STAGE03A_MATRIX", "")),
    # Prefer model-scoped output folder so other GLOBAL_DESCRIPTORs do not collide.
    REPO_ROOT / "outputs" / "stage03" / "03a" / GLOBAL_DESCRIPTOR / DATASET_NAME / artifact_name,
    HERE / "outputs" / "stage03" / "03a" / GLOBAL_DESCRIPTOR / DATASET_NAME / artifact_name,
    HERE / artifact_name,
    HERE / "notebooks" / artifact_name,
    REPO_ROOT / "notebooks" / artifact_name,
    Path(f"/kaggle/working/apexid_cvpr_stage03/03a/{GLOBAL_DESCRIPTOR}/{DATASET_NAME}/{artifact_name}"),
    Path(f"/kaggle/input/datasets/mohamedbassat/apexid_cvpr_stage03/03a/{GLOBAL_DESCRIPTOR}/{DATASET_NAME}/{artifact_name}"),
]
for parent in (Path("/kaggle/input/datasets/mohamedbassat"), Path("/kaggle/input")):
    if not parent.is_dir():
        continue
    for child in parent.iterdir():
        if not child.is_dir():
            continue
        STAGE03A_MATRIX_CANDIDATES.extend(
            [
                child / artifact_name,
                child / GLOBAL_DESCRIPTOR / DATASET_NAME / artifact_name,
                child / "03a" / GLOBAL_DESCRIPTOR / DATASET_NAME / artifact_name,
                child / "apexid_cvpr_stage03" / "03a" / GLOBAL_DESCRIPTOR / DATASET_NAME / artifact_name,
            ]
        )
MATRIX_PATH = next((p.resolve() for p in STAGE03A_MATRIX_CANDIDATES if str(p) and p.is_file()), None)
if MATRIX_PATH is None:
    raise FileNotFoundError(
        f"Attach or place {artifact_name} under outputs/stage03/03a/{GLOBAL_DESCRIPTOR}/{DATASET_NAME}/."
    )
local_output = REPO_ROOT / "outputs" / "stage03" / "03b" / GLOBAL_DESCRIPTOR / DATASET_NAME
OUTPUT_ROOT = Path(os.environ.get(
    "APEXID_STAGE03B_OUTPUT",
    f"/kaggle/working/apexid_cvpr_stage03/03b/{GLOBAL_DESCRIPTOR}/{DATASET_NAME}" if ON_KAGGLE
    else str(local_output),
)).resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("MATRIX_PATH:", MATRIX_PATH)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("STAGE03_SPLITS_ROOT:", next((p for p in STAGE03_SPLITS_ROOT_CANDIDATES if str(p) and p.is_dir()), STAGE03_SPLITS_ROOT_CANDIDATES[1]))
print("PROTOCOL: decided from 03a artifact after load")

Note: you may need to restart the kernel to use updated packages.
MATRIX_PATH: C:\Users\Legion 5\Desktop\ApexID\notebooks\sim_matrix_ApexID_BHCA_A_v2_bristol_gorillas_open.npy
OUTPUT_ROOT: C:\Users\Legion 5\Desktop\ApexID\outputs\stage03\03b\ApexID_BHCA_A_v2\bristol_gorillas
STAGE03_SPLITS_ROOT: .
PROTOCOL: decided from 03a artifact after load


In [8]:
# Load the Stage 03a query-query / query-gallery artifact. Do not rebuild the split.
obj = np.load(MATRIX_PATH, allow_pickle=True).item()
if DATASET_NAME not in obj:
    raise KeyError(f"{MATRIX_PATH} has keys {list(obj)}; expected {DATASET_NAME}")
bundle = obj[DATASET_NAME]
if bundle.get("global_descriptor") != GLOBAL_DESCRIPTOR:
    raise ValueError(f"Artifact model mismatch: {bundle.get('global_descriptor')} != {GLOBAL_DESCRIPTOR}")
if bundle.get("dataset") != DATASET_NAME:
    raise ValueError("03a dataset does not match the 03b knob")
PROTOCOL = str(bundle.get("protocol") or "")
if PROTOCOL not in {"gallery_attach_optuna", "query_only_defaults"}:
    raise ValueError(f"Unknown 03a protocol {PROTOCOL!r}")
print("PROTOCOL from 03a:", PROTOCOL)

sim_qq = np.asarray(bundle["sim_query_query"], dtype=np.float32)
sim_qg = np.asarray(bundle.get("sim_query_db", np.zeros((len(sim_qq), 0), dtype=np.float32)), dtype=np.float32)
query_image_ids = np.asarray(bundle["query_image_ids"]).astype(str)
db_image_ids = np.asarray(bundle["db_image_ids"]).astype(str)
y_query = np.asarray(bundle["query_identities"]).astype(str)
y_db = np.asarray(bundle["db_identities"]).astype(str)
query_open_role = np.asarray(bundle["query_open_role"]).astype(str)
if sim_qq.shape != (len(query_image_ids), len(query_image_ids)):
    raise ValueError("Query-query matrix does not match query_image_ids")
if sim_qg.shape != (len(query_image_ids), len(db_image_ids)):
    raise ValueError("Query-gallery matrix does not match query/gallery ids")
if not np.isfinite(sim_qq).all() or sim_qq.min() < 0 or sim_qq.max() > 1:
    raise ValueError("Query-query similarities must be finite and clipped to [0,1]")
if sim_qg.size and (not np.isfinite(sim_qg).all() or sim_qg.min() < 0 or sim_qg.max() > 1):
    raise ValueError("Query-gallery similarities must be finite and clipped to [0,1]")
if len(set(query_image_ids)) != len(query_image_ids) or len(set(db_image_ids)) != len(db_image_ids):
    raise ValueError("03a image_ids are not unique")
if set(query_image_ids) & set(db_image_ids):
    raise RuntimeError("Query/gallery image leakage in the 03a artifact")

split = None
split_path = None
if PROTOCOL == "gallery_attach_optuna":
    known_mask = query_open_role == "query_known"
    novel_mask = query_open_role == "query_novel"
    if not known_mask.any() or not novel_mask.any():
        raise RuntimeError("Open split must contain known and novel queries")
    if not len(db_image_ids):
        raise RuntimeError("Standard protocol requires a gallery")
    if set(y_query[novel_mask]) & set(y_db):
        raise RuntimeError("Unknown-query identities leaked into the gallery")
    for identity in np.unique(y_query[known_mask]):
        if identity not in set(y_db):
            raise RuntimeError(f"Known identity {identity!r} lacks a gallery sample")
    split = {
        "sim_query_query": sim_qq,
        "sim_query_db": sim_qg,
        "y_true_query": y_query,
        "y_true_db": y_db,
        "query_image_ids": query_image_ids,
        "db_image_ids": db_image_ids,
        "known_mask_query": known_mask,
        "unknown_mask_query": novel_mask,
    }
    split_path = OUTPUT_ROOT / f"optuna_split_{GLOBAL_DESCRIPTOR}_{DATASET_NAME}.npz"
    np.savez_compressed(split_path, **{key: value for key, value in split.items() if isinstance(value, np.ndarray)})
    print("Query-query:", sim_qq.shape, "query-gallery:", sim_qg.shape)
    print("Queries:", len(query_image_ids), "gallery:", len(db_image_ids))
else:
    if sim_qg.shape[1] != 0:
        raise RuntimeError("Gallery-free query-gallery matrix must be empty")
    print(f"{DATASET_NAME} gallery-free protocol: using 03a query-query only. No Optuna split.")


PROTOCOL from 03a: gallery_attach_optuna
Query-query: (1789, 1789) query-gallery: (1789, 3535)
Queries: 1789 gallery: 3535


In [9]:
# AnimalCLEF identity scoring + graph clustering (ported from MIA-AI-Team/AnimalCLEF26).
def compute_identity_scores(sim_query_db, db_ids, top_r):
    identity_names = list(dict.fromkeys(db_ids.tolist()))
    scores = np.zeros((sim_query_db.shape[0], len(identity_names)), dtype=np.float32)
    for j, identity in enumerate(identity_names):
        cols = np.flatnonzero(db_ids == identity)
        sims = sim_query_db[:, cols]
        r = min(int(top_r), sims.shape[1])
        scores[:, j] = sims.mean(axis=1) if r == sims.shape[1] else np.partition(sims, -r, axis=1)[:, -r:].mean(axis=1)
    return np.asarray(identity_names, dtype=object), scores


def summarize_best_identities(names, scores):
    best_idx = scores.argmax(axis=1)
    best_scores = scores[np.arange(len(scores)), best_idx]
    best_names = names[best_idx]
    if scores.shape[1] >= 2:
        top2 = np.partition(scores, -2, axis=1)[:, -2:]
        second = np.min(top2, axis=1)
    else:
        second = np.zeros(len(scores), dtype=np.float32)
    return best_names, best_scores, second, best_scores - second


def get_topk_neighbors(sim_matrix, top_k):
    n = sim_matrix.shape[0]
    k = min(int(top_k), n - 1)
    if k <= 0:
        return np.empty((n, 0), dtype=np.int64)
    sims = sim_matrix.copy()
    np.fill_diagonal(sims, -np.inf)
    return np.argpartition(sims, -k, axis=1)[:, -k:]


def chinese_whispers(sim_matrix, threshold, iterations, top_k, k_core_degree, seed=42, skip_mutual_knn=False):
    random.seed(seed)
    np.random.seed(seed)
    n = sim_matrix.shape[0]
    adj = sim_matrix.copy()
    np.fill_diagonal(adj, 0)
    adj[adj < threshold] = 0
    graph = nx.from_numpy_array(adj)
    if graph.number_of_edges():
        try:
            backbone = nx.k_core(graph, k=int(k_core_degree))
            if backbone.number_of_nodes() == 0:
                backbone = graph
        except Exception:
            backbone = graph
    else:
        backbone = graph
    backbone_nodes = list(backbone.nodes()) or list(range(n))
    in_backbone = np.zeros(n, dtype=bool)
    in_backbone[backbone_nodes] = True
    valid = (sim_matrix >= threshold) & in_backbone[:, None] & in_backbone[None, :]
    np.fill_diagonal(valid, False)
    if not skip_mutual_knn:
        topk = get_topk_neighbors(sim_matrix, top_k)
        nearest = np.zeros((n, n), dtype=bool)
        if topk.shape[1]:
            nearest[np.arange(n)[:, None], topk] = True
        valid &= nearest & nearest.T
    sources, targets = np.nonzero(valid)
    graph_list = [[] for _ in range(n)]
    for source, target in zip(sources, targets):
        graph_list[int(source)].append((int(target), float(sim_matrix[source, target])))
    labels = list(range(n))
    nodes = backbone_nodes.copy()
    for _ in range(int(iterations)):
        random.shuffle(nodes)
        changes = 0
        for node in nodes:
            weights = defaultdict(float)
            for neighbor, weight in graph_list[node]:
                weights[labels[neighbor]] += weight
            if weights:
                new_label = max(weights.items(), key=lambda item: (item[1], -item[0]))[0]
                changes += int(new_label != labels[node])
                labels[node] = new_label
        if changes == 0:
            break
    return np.unique(labels, return_inverse=True)[1].astype(np.int64)


def final_open_set_labels(cluster_ids, best_names, best_scores, margins, tau_attach, tau_margin):
    votes = defaultdict(lambda: defaultdict(float))
    for i, cluster in enumerate(cluster_ids):
        if best_scores[i] >= tau_attach and margins[i] >= tau_margin:
            votes[int(cluster)][str(best_names[i])] += float(best_scores[i])
    cluster_to_known = {
        cluster: max(identity_votes.items(), key=lambda item: item[1])[0]
        for cluster, identity_votes in votes.items() if identity_votes
    }
    return np.asarray([
        cluster_to_known.get(int(cluster), f"__new_cluster_{int(cluster)}")
        for cluster in cluster_ids
    ], dtype=str)

In [10]:
# Exact k-reciprocal implementation used by the AnimalCLEF repository.
def re_ranking(sim_matrix, k1=20, k2=6, lambda_value=0.3):
    sim_matrix = np.asarray(sim_matrix, dtype=np.float32)
    n = sim_matrix.shape[0]
    original_dist = np.clip(1.0 - sim_matrix, 0.0, None)
    col_max = np.max(original_dist, axis=0)
    col_max[col_max == 0] = 1.0
    original_dist = (original_dist / col_max).T.astype(np.float32)
    values = np.zeros((n, n), dtype=np.float32)
    initial_rank = np.argsort(original_dist, axis=1).astype(np.int32)
    half_k1 = int(np.around(k1 / 2.0)) + 1
    for i in range(n):
        forward = initial_rank[i, : k1 + 1]
        backward = initial_rank[forward, : k1 + 1]
        reciprocal = forward[np.where(backward == i)[0]]
        expanded = reciprocal.copy()
        for q in reciprocal:
            q_forward = initial_rank[q, :half_k1]
            q_backward = initial_rank[q_forward, :half_k1]
            q_recip = q_forward[np.where(q_backward == q)[0]]
            if len(q_recip) and len(np.intersect1d(q_recip, reciprocal)) > (2.0 / 3.0) * len(q_recip):
                expanded = np.append(expanded, q_recip)
        expanded = np.unique(expanded)
        weight = np.exp(-original_dist[i, expanded])
        if weight.sum() > 0:
            values[i, expanded] = weight / weight.sum()
    if k2 > 1:
        expanded_values = np.zeros_like(values)
        for i in range(n):
            expanded_values[i] = np.mean(values[initial_rank[i, :k2]], axis=0)
        values = expanded_values
    jaccard = np.zeros((n, n), dtype=np.float32)
    inverse = [np.where(values[:, j] != 0)[0] for j in range(n)]
    for i in range(n):
        temp_min = np.zeros(n, dtype=np.float32)
        nonzero = np.where(values[i] != 0)[0]
        for column in nonzero:
            rows = inverse[column]
            temp_min[rows] += np.minimum(values[i, column], values[rows, column])
        jaccard[i] = 1.0 - temp_min / (2.0 - temp_min + 1e-12)
    final_dist = np.clip(jaccard * (1.0 - lambda_value) + original_dist * lambda_value, 0.0, 1.0)
    np.fill_diagonal(final_dist, 0.0)
    return (1.0 - final_dist).astype(np.float32)

In [11]:
study = None
sim_qq = None
sim_qg = None
y_query = None
y_db = None

if PROTOCOL == "gallery_attach_optuna":
    sim_qq = split["sim_query_query"]
    sim_qg = split["sim_query_db"]
    y_query = split["y_true_query"]
    y_db = split["y_true_db"]

    def objective(trial):
        top_r = trial.suggest_int("top_r", 1, 5)
        tau_attach = trial.suggest_float("tau_attach", 0.10, 0.90, step=0.01)
        tau_margin = trial.suggest_float("tau_margin", 0.10, 0.90, step=0.01)
        cw_threshold = trial.suggest_float("cw_threshold", 0.12, 0.68, step=0.01)
        cw_top_k = trial.suggest_int("cw_top_k", 7, 60)
        cw_iterations = trial.suggest_int("cw_iterations", 5, 30)
        cw_k_core_degree = trial.suggest_int("cw_k_core_degree", 1, 5)
        rr_k1 = trial.suggest_int("rr_k1", 5, 50)
        rr_k2 = trial.suggest_int("rr_k2", 1, 15)
        rr_lambda = trial.suggest_float("rr_lambda", 0.10, 0.90)

        identity_names, identity_scores = compute_identity_scores(sim_qg, y_db, top_r)
        best_names, best_scores, _, margins = summarize_best_identities(identity_names, identity_scores)
        sim_for_cw = re_ranking(sim_qq, rr_k1, rr_k2, rr_lambda)
        clusters = chinese_whispers(
            sim_for_cw,
            threshold=cw_threshold,
            iterations=cw_iterations,
            top_k=cw_top_k,
            k_core_degree=cw_k_core_degree,
            seed=42,
            skip_mutual_knn=True,
        )
        predicted = final_open_set_labels(clusters, best_names, best_scores, margins, tau_attach, tau_margin)
        ari = adjusted_rand_score(y_query, predicted)
        v_measure = v_measure_score(y_query, predicted)
        trial.set_user_attr("ari", float(ari))
        trial.set_user_attr("v_measure", float(v_measure))
        return 0.6 * ari + 0.4 * v_measure

    storage_path = OUTPUT_ROOT / f"optuna_{GLOBAL_DESCRIPTOR}_{DATASET_NAME}.sqlite3"
    study = optuna.create_study(
        study_name=f"{GLOBAL_DESCRIPTOR}__{DATASET_NAME}",
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=OPTUNA_SEED, multivariate=True),
        storage=f"sqlite:///{storage_path.as_posix()}",
        load_if_exists=True,
    )
    completed_trials = sum(trial.state == optuna.trial.TrialState.COMPLETE for trial in study.trials)
    remaining = max(0, N_TRIALS - completed_trials)
    print("Completed trials:", completed_trials, "remaining:", remaining)
    if remaining:
        study.optimize(objective, n_trials=remaining, gc_after_trial=True, show_progress_bar=True)
    print("Best value:", study.best_value)
    print("Best params:", study.best_params)
else:
    print("Optuna skipped. Frozen Jaguar query-only defaults:", QUERY_ONLY_DEFAULTS)

[I 2026-09-24 15:29:53,528] A new study created in RDB with name: ApexID_BHCA_A_v2__bristol_gorillas


Completed trials: 0 remaining: 2000


Best trial: 0. Best value: 0.198657:   0%|          | 1/2000 [00:03<1:57:23,  3.52s/it]

[I 2026-09-24 15:29:56,942] Trial 0 finished with value: 0.19865658100390388 and parameters: {'top_r': 2, 'tau_attach': 0.87, 'tau_margin': 0.69, 'cw_threshold': 0.46, 'cw_top_k': 15, 'cw_iterations': 9, 'cw_k_core_degree': 1, 'rr_k1': 44, 'rr_k2': 10, 'rr_lambda': 0.6664580622368363}. Best is trial 0 with value: 0.19865658100390388.


Best trial: 1. Best value: 0.232145:   0%|          | 2/2000 [00:05<1:21:35,  2.45s/it]

[I 2026-09-24 15:29:58,653] Trial 1 finished with value: 0.2321449636053181 and parameters: {'top_r': 1, 'tau_attach': 0.88, 'tau_margin': 0.77, 'cw_threshold': 0.24, 'cw_top_k': 16, 'cw_iterations': 9, 'cw_k_core_degree': 2, 'rr_k1': 29, 'rr_k2': 7, 'rr_lambda': 0.3329833121584336}. Best is trial 1 with value: 0.2321449636053181.


... (3,992 outputs left out)

Best trial: 1747. Best value: 0.289387: 100%|█████████▉| 1999/2000 [2:13:42<00:03,  3.74s/it]

[I 2026-09-24 17:43:35,556] Trial 1998 finished with value: 0.2623403990810942 and parameters: {'top_r': 5, 'tau_attach': 0.83, 'tau_margin': 0.21000000000000002, 'cw_threshold': 0.12, 'cw_top_k': 18, 'cw_iterations': 13, 'cw_k_core_degree': 1, 'rr_k1': 50, 'rr_k2': 13, 'rr_lambda': 0.44071262275601775}. Best is trial 1747 with value: 0.28938726066971465.


Best trial: 1747. Best value: 0.289387: 100%|██████████| 2000/2000 [2:13:46<00:00,  4.01s/it]

[I 2026-09-24 17:43:39,524] Trial 1999 finished with value: 0.27314515041157617 and parameters: {'top_r': 5, 'tau_attach': 0.78, 'tau_margin': 0.25, 'cw_threshold': 0.13999999999999999, 'cw_top_k': 15, 'cw_iterations': 8, 'cw_k_core_degree': 1, 'rr_k1': 50, 'rr_k2': 14, 'rr_lambda': 0.5260453641598959}. Best is trial 1747 with value: 0.28938726066971465.
Best value: 0.28938726066971465
Best params: {'top_r': 5, 'tau_attach': 0.84, 'tau_margin': 0.2, 'cw_threshold': 0.13, 'cw_top_k': 18, 'cw_iterations': 15, 'cw_k_core_degree': 1, 'rr_k1': 50, 'rr_k2': 13, 'rr_lambda': 0.49561469479285564}


In [12]:
# Freeze either the independently reproduced Optuna winner or the label-free defaults.
if PROTOCOL == "gallery_attach_optuna":
    best_params = dict(study.best_params)
    best_params["use_rerank"] = USE_RERANK

    identity_names, identity_scores = compute_identity_scores(sim_qg, y_db, int(best_params["top_r"]))
    best_names, best_scores, _, margins = summarize_best_identities(identity_names, identity_scores)
    best_sim_for_cw = re_ranking(
        sim_qq,
        int(best_params["rr_k1"]),
        int(best_params["rr_k2"]),
        float(best_params["rr_lambda"]),
    )
    best_clusters = chinese_whispers(
        best_sim_for_cw,
        threshold=float(best_params["cw_threshold"]),
        iterations=int(best_params["cw_iterations"]),
        top_k=int(best_params["cw_top_k"]),
        k_core_degree=int(best_params["cw_k_core_degree"]),
        seed=42,
        skip_mutual_knn=True,
    )
    best_predicted = final_open_set_labels(
        best_clusters, best_names, best_scores, margins,
        float(best_params["tau_attach"]), float(best_params["tau_margin"]),
    )
    recomputed_ari = float(adjusted_rand_score(y_query, best_predicted))
    recomputed_v_measure = float(v_measure_score(y_query, best_predicted))
    recomputed_objective = 0.6 * recomputed_ari + 0.4 * recomputed_v_measure
    if not np.isclose(recomputed_objective, float(study.best_value), rtol=0.0, atol=1e-12):
        raise RuntimeError(
            f"Winning trial is not reproducible: stored={study.best_value}, recomputed={recomputed_objective}"
        )

    completed_trials = sum(trial.state == optuna.trial.TrialState.COMPLETE for trial in study.trials)
    best_config = {
        **best_params,
        "protocol": PROTOCOL,
        "config_source": "optuna_stage03_open_split",
        "objective": "0.6*ARI+0.4*V-measure",
        "objective_value": recomputed_objective,
        "validation_ari": recomputed_ari,
        "validation_v_measure": recomputed_v_measure,
        "n_trials_completed": int(completed_trials),
        "split_seed": SPLIT_SEED,
        "known_identity_frac": KNOWN_IDENTITY_FRAC,
        "db_frac_within_known": DB_FRAC_WITHIN_KNOWN,
        "min_images_per_known_identity": MIN_IMAGES_PER_KNOWN_IDENTITY,
        "dataset": DATASET_NAME,
        "global_descriptor": GLOBAL_DESCRIPTOR,
    }
    study.trials_dataframe().to_csv(OUTPUT_ROOT / "optuna_trials.csv", index=False)
    split_sha256 = hashlib.sha256(split_path.read_bytes()).hexdigest()
    winning_trial_recomputed = True
else:
    best_config = {
        **QUERY_ONLY_DEFAULTS,
        "protocol": PROTOCOL,
        "config_source": "frozen_label_free_defaults",
        "tuning_performed": False,
        "n_trials_completed": 0,
        "database_required": False,
        "known_attachment_enabled": False,
        "cw_seed": 42,
        "dataset": DATASET_NAME,
        "global_descriptor": GLOBAL_DESCRIPTOR,
    }
    split_sha256 = None
    winning_trial_recomputed = False

config_path = OUTPUT_ROOT / f"best_configs_{GLOBAL_DESCRIPTOR}_{DATASET_NAME}.json"
config_path.write_text(json.dumps({DATASET_NAME: best_config}, indent=2, sort_keys=True), encoding="utf-8")
manifest = {
    "stage": "03b_frozen_clustering_config",
    "status": "passed",
    "protocol": PROTOCOL,
    "dataset": DATASET_NAME,
    "global_descriptor": GLOBAL_DESCRIPTOR,
    "source_matrix": str(MATRIX_PATH),
    "source_matrix_sha256": hashlib.sha256(MATRIX_PATH.read_bytes()).hexdigest(),
    "optuna_split_sha256": split_sha256,
    "stage00_test_split_used_for_tuning": False,
    "stage03_open_split_used_for_tuning": True,
    "winning_trial_recomputed": winning_trial_recomputed,
    "rerank_mutual_knn_policy": "skip_mutual_knn=True after reranking",
    "config": best_config,
}
(OUTPUT_ROOT / "stage03b_manifest.json").write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")
print(json.dumps({DATASET_NAME: best_config}, indent=2))
print(f"\nNEXT: Save {OUTPUT_ROOT} as a Kaggle dataset and attach it with the 03a matrix to notebook 03c.")

{
  "bristol_gorillas": {
    "top_r": 5,
    "tau_attach": 0.84,
    "tau_margin": 0.2,
    "cw_threshold": 0.13,
    "cw_top_k": 18,
    "cw_iterations": 15,
    "cw_k_core_degree": 1,
    "rr_k1": 50,
    "rr_k2": 13,
    "rr_lambda": 0.49561469479285564,
    "use_rerank": true,
    "protocol": "gallery_attach_optuna",
    "config_source": "optuna_stage03_open_split",
    "objective": "0.6*ARI+0.4*V-measure",
    "objective_value": 0.28938726066971465,
    "validation_ari": 0.1691367413161989,
    "validation_v_measure": 0.46976303969998834,
    "n_trials_completed": 2000,
    "split_seed": 42,
    "known_identity_frac": 0.8,
    "db_frac_within_known": 0.8,
    "min_images_per_known_identity": 3,
    "dataset": "bristol_gorillas",
    "global_descriptor": "ApexID_BHCA_A_v2"
  }
}

NEXT: Save C:\Users\Legion 5\Desktop\ApexID\outputs\stage03\03b\ApexID_BHCA_A_v2\bristol_gorillas as a Kaggle dataset and attach it with the 03a matrix to notebook 03c.
